In [ ]:

import numpy as np
from pathlib import Path
import json
import joblib

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

import xgboost as xgb  # add near your imports (top of notebook)



import optuna
print(optuna.__version__)

4.7.0


c:\Users\aadha\anaconda3\envs\f1ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
from pathlib import Path
import fastf1

PROJECT_ROOT = Path.cwd().parents[1]  # adjust if needed
CACHE_DIR = PROJECT_ROOT / "data_raw" / "f1_cache"

fastf1.Cache.enable_cache(str(CACHE_DIR.resolve()))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CACHE_DIR:", CACHE_DIR.resolve())




ART_DIR = Path.cwd() / "artifacts"
ART_DIR.mkdir(exist_ok=True)

print("ART_DIR:", ART_DIR.resolve())

PROJECT_ROOT: c:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory
CACHE_DIR: C:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory\data_raw\f1_cache
ART_DIR: C:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory\notebooks\Phase 3\artifacts


In [21]:

import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import fastf1



# --- experiment split ---
TRAIN_YEARS = list(range(2018, 2024))  # 2018..2023
EVAL_YEAR = 2024
TEST_YEAR = 2025

YEARS = TRAIN_YEARS + [EVAL_YEAR, TEST_YEAR]
SESSION = "Q"

TARGET = "quali_position"
ID_COLS = ["year", "round", "event_name",  "driver"]

print("Cache:", CACHE_DIR.resolve())
print("Years:", YEARS)

for y in YEARS:
    sched = fastf1.get_event_schedule(y)
    print(y, "schedule rows:", len(sched), "| cols:", list(sched.columns))

Cache: C:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory\data_raw\f1_cache
Years: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
2018 schedule rows: 21 | cols: ['RoundNumber', 'Country', 'Location', 'OfficialEventName', 'EventDate', 'EventName', 'EventFormat', 'Session1', 'Session1Date', 'Session1DateUtc', 'Session2', 'Session2Date', 'Session2DateUtc', 'Session3', 'Session3Date', 'Session3DateUtc', 'Session4', 'Session4Date', 'Session4DateUtc', 'Session5', 'Session5Date', 'Session5DateUtc', 'F1ApiSupport']
2019 schedule rows: 21 | cols: ['RoundNumber', 'Country', 'Location', 'OfficialEventName', 'EventDate', 'EventName', 'EventFormat', 'Session1', 'Session1Date', 'Session1DateUtc', 'Session2', 'Session2Date', 'Session2DateUtc', 'Session3', 'Session3Date', 'Session3DateUtc', 'Session4', 'Session4Date', 'Session4DateUtc', 'Session5', 'Session5Date', 'Session5DateUtc', 'F1ApiSupport']
2020 schedule rows: 19 | cols: ['RoundNumber

In [22]:
import pandas as pd
import numpy as np
import fastf1

FEAT_PATH = ART_DIR / "df_feat.parquet"

feat_rows = []
fail_feat = []

for year in YEARS:
    try:
        sched = fastf1.get_event_schedule(year)
    except Exception as e:
        fail_feat.append((year, "schedule", type(e).__name__, str(e)))
        continue

    for _, ev in sched.iterrows():
        rnd = ev.get("RoundNumber")
        if pd.isna(rnd) or int(rnd) <= 0:
            continue
        rnd = int(rnd)

        event_name = ev.get("EventName")
        country = ev.get("Country")
        location = ev.get("Location")

        try:
            ses = fastf1.get_session(year, rnd, SESSION)

            # ✅ THIS is the big fix: you need laps=True for feature building
            ses.load(laps=True, telemetry=False, weather=False, messages=False)

            laps = ses.laps
            if laps is None or laps.empty:
                fail_feat.append((year, rnd, "EmptyLaps", event_name))
                continue

            # Standardize driver id to DriverNumber (matches well with results)
            if "DriverNumber" not in laps.columns:
                fail_feat.append((year, rnd, "NoDriverNumberInLaps", event_name))
                continue

            # Keep only "real" laps
            laps = laps.dropna(subset=["LapTime"]).copy()

            # Build a small but solid feature set per driver
            g = laps.groupby("DriverNumber", dropna=False)

            df_g = pd.DataFrame({
                "driver": g.size().index.astype(str),

                "n_laps": g.size().values,

                "best_laptime_s": g["LapTime"].min().dt.total_seconds().values,
                "avg_laptime_s": g["LapTime"].mean().dt.total_seconds().values,

                "best_s1_s": g["Sector1Time"].min().dt.total_seconds().values if "Sector1Time" in laps else np.nan,
                "best_s2_s": g["Sector2Time"].min().dt.total_seconds().values if "Sector2Time" in laps else np.nan,
                "best_s3_s": g["Sector3Time"].min().dt.total_seconds().values if "Sector3Time" in laps else np.nan,
            })

            df_g["year"] = year
            df_g["round"] = rnd
            df_g["event_name"] = event_name
            df_g["country"] = country
            df_g["location"] = location

            feat_rows.append(df_g)

        except Exception as e:
            fail_feat.append((year, rnd, type(e).__name__, str(e)))

df_feat = pd.concat(feat_rows, ignore_index=True) if feat_rows else pd.DataFrame()

print("df_feat:", df_feat.shape, "| failures:", len(fail_feat))
print("save ->", FEAT_PATH)
df_feat.to_parquet(FEAT_PATH, index=False)

assert not df_feat.empty, "df_feat is EMPTY — feature builder failed"
df_feat.head()
print("df_feat shape:", df_feat.shape)

assert not df_feat.empty, "df_feat is EMPTY — feature builder failed"

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['44', '7', '5', '33', '3', '20', '8', '27', '55', '77', '14', '2', '11', '18', '31', '28', '9', '16', '35', '10']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Usi

df_feat: (3432, 12) | failures: 0
save -> c:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory\notebooks\Phase 3\artifacts\df_feat.parquet
df_feat shape: (3432, 12)


In [23]:
from pathlib import Path
import pandas as pd

print("CWD:", Path.cwd().resolve())

p = Path("artifacts") / "df_feat.parquet"
print("Relative parquet path resolves to:", p.resolve())
print("Exists?", p.exists(), "Size:", p.stat().st_size if p.exists() else None)

df_feat = pd.read_parquet(p)
print("Loaded df_feat shape:", df_feat.shape)
print("Columns:", list(df_feat.columns))
df_feat.head()

CWD: C:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory\notebooks\Phase 3
Relative parquet path resolves to: C:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory\notebooks\Phase 3\artifacts\df_feat.parquet
Exists? True Size: 129760
Loaded df_feat shape: (3432, 12)
Columns: ['driver', 'n_laps', 'best_laptime_s', 'avg_laptime_s', 'best_s1_s', 'best_s2_s', 'best_s3_s', 'year', 'round', 'event_name', 'country', 'location']


,driver,n_laps,best_laptime_s,avg_laptime_s,best_s1_s,best_s2_s,best_s3_s,year,round,event_name,country,location
0,10,5,85.295,103.418000,28.146,23.144,33.959,2018,1,Australian Grand Prix,Australia,Melbourne
1,11,10,84.005,95.566200,27.618,22.786,33.518,2018,1,Australian Grand Prix,Australia,Melbourne
2,14,10,83.597,96.737200,27.489,22.647,33.206,2018,1,Australian Grand Prix,Australia,Melbourne
3,16,6,84.636,96.620500,28.153,22.901,33.582,2018,1,Australian Grand Prix,Australia,Melbourne
4,18,9,84.230,98.978333,27.793,22.771,33.657,2018,1,Australian Grand Prix,Australia,Melbourne


In [24]:
# Cell 2 — Label builder df_lab (CACHE_ONLY-safe)

label_rows = []
fail_lab = []

# OPTION 1 (recommended): provide your own rounds per year.
# If you already have SG_ROUNDS elsewhere, use that and delete this dict.
# Example below assumes you only care about Singapore rounds.
SG_ROUNDS = {
    # year: round_number
    2018: 15,
    2019: 15,
    2022: 17,
    2023: 15,
    2024: 18,
    2025: 18,
    # add others you actually have cached
}

for year in YEARS:
    # Build the list of rounds without calling get_event_schedule (no internet needed)
    rounds = []
    if year in SG_ROUNDS:
        rounds = [int(SG_ROUNDS[year])]
    else:
        fail_lab.append((year, "schedule", "NoRoundMapping", "No round mapping for this year"))
        continue

    for rnd in rounds:
        try:
            ses = fastf1.get_session(year, rnd, SESSION)
            ses.load(laps=False, telemetry=False, weather=False, messages=False)

            # Metadata (safe even if schedule isn't fetched)
            event_name = getattr(ses.event, "EventName", None)
            country    = getattr(ses.event, "Country", None)
            location   = getattr(ses.event, "Location", None)

            res = ses.results
            if res is None or res.empty:
                fail_lab.append((year, rnd, "EmptyResults", event_name))
                continue

            driver_col = "DriverNumber" if "DriverNumber" in res.columns else "Abbreviation"

            for _, r in res.iterrows():
                label_rows.append({
                    "year": year,
                    "round": rnd,
                    "event_name": event_name,
                    "country": country,
                    "location": location,
                    "driver": str(r.get(driver_col, None)),
                    TARGET: r.get("Position", None),
                })

        except Exception as e:
            fail_lab.append((year, rnd, type(e).__name__, str(e)))

df_lab = pd.DataFrame(label_rows)
print("df_lab:", df_lab.shape, "| failures:", len(fail_lab))
df_lab.head()

core           INFO 	Loading data for Singapore Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info


req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '5', '77', '7', '3', '11', '8', '31', '27', '14', '55', '16', '9', '10', '20', '28', '2', '35', '18']
core           INFO 	Loading data for Singapore Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['16', '44', '5', '33', '77', '23', '55', '27', '4', '11', '99', '10', '7', '20', '26', '18', '8', '63', '88', '3']
core           INFO 	Loading data for Singapore Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['16', '11', '44', '55', '14', '4', '10', '1', '20', '22', '63', '18', '47', '5', '24', '77', '3', '31', '23', '6']
core           INFO 	Loading data for

df_lab: (120, 7) | failures: 2


,year,round,event_name,country,location,driver,quali_position
0,2018,15,Singapore Grand Prix,Singapore,Singapore,44,1.0
1,2018,15,Singapore Grand Prix,Singapore,Singapore,33,2.0
2,2018,15,Singapore Grand Prix,Singapore,Singapore,5,3.0
3,2018,15,Singapore Grand Prix,Singapore,Singapore,77,4.0
4,2018,15,Singapore Grand Prix,Singapore,Singapore,7,5.0


In [25]:
# ==========================
# Cell 2.5 — Load df_feat artifact
# ==========================
from pathlib import Path

FEAT_PATH = Path("artifacts") / "df_feat.parquet"

if not FEAT_PATH.exists():
    raise RuntimeError("df_feat artifact not found. Run feature builder first.")

df_feat = pd.read_parquet(FEAT_PATH)

print("[df_feat] loaded")
print("[df_feat] shape:", df_feat.shape)
print("[df_feat] columns:", list(df_feat.columns))

[df_feat] loaded
[df_feat] shape: (3432, 12)
[df_feat] columns: ['driver', 'n_laps', 'best_laptime_s', 'avg_laptime_s', 'best_s1_s', 'best_s2_s', 'best_s3_s', 'year', 'round', 'event_name', 'country', 'location']


In [26]:
# ==========================
# Cell 3 — Merge + clean
# ==========================
df = df_feat.merge(df_lab, on=["year", "round", "event_name", "driver"], how="inner")

df_model = df.dropna(subset=[TARGET]).copy()
df_model[TARGET] = pd.to_numeric(df_model[TARGET], errors="coerce").astype(int)

print("merged df:", df.shape)
print("df_model:", df_model.shape)
print("years:", sorted(df_model["year"].unique()))
df_model[ID_COLS + [TARGET]].head()

merged df: (120, 15)
df_model: (120, 15)
years: [np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,year,round,event_name,driver,quali_position
0,2018,15,Singapore Grand Prix,10,15
1,2018,15,Singapore Grand Prix,11,7
2,2018,15,Singapore Grand Prix,14,11
3,2018,15,Singapore Grand Prix,16,13
4,2018,15,Singapore Grand Prix,18,20


In [27]:
#debugg
# Debug: check merge keys exist
print("df_feat cols:", list(df_feat.columns))
print("df_lab  cols:", list(df_lab.columns))

keys = ["year", "round", "event_name", "driver"]
print("\nMissing in df_feat:", [k for k in keys if k not in df_feat.columns])
print("Missing in df_lab :", [k for k in keys if k not in df_lab.columns])

print("\nHead df_feat keys:")
display(df_feat[[c for c in keys if c in df_feat.columns]].head())

print("\nHead df_lab keys:")
display(df_lab[[c for c in keys if c in df_lab.columns]].head())

df_feat cols: ['driver', 'n_laps', 'best_laptime_s', 'avg_laptime_s', 'best_s1_s', 'best_s2_s', 'best_s3_s', 'year', 'round', 'event_name', 'country', 'location']
df_lab  cols: ['year', 'round', 'event_name', 'country', 'location', 'driver', 'quali_position']

Missing in df_feat: []
Missing in df_lab : []

Head df_feat keys:


,year,round,event_name,driver
0,2018,1,Australian Grand Prix,10
1,2018,1,Australian Grand Prix,11
2,2018,1,Australian Grand Prix,14
3,2018,1,Australian Grand Prix,16
4,2018,1,Australian Grand Prix,18



Head df_lab keys:


,year,round,event_name,driver
0,2018,15,Singapore Grand Prix,44
1,2018,15,Singapore Grand Prix,33
2,2018,15,Singapore Grand Prix,5
3,2018,15,Singapore Grand Prix,77
4,2018,15,Singapore Grand Prix,7


In [28]:
# ==================================
# Cell 4 — Build X/y splits by years
# ==================================
# Numeric feature columns = everything except ids + target
drop_cols = set(ID_COLS + [TARGET])
num_cols = [c for c in df_model.columns if c not in drop_cols]

# Keep only numeric columns (defensive)
X_all = df_model[num_cols].apply(pd.to_numeric, errors="coerce")
y_all = df_model[TARGET].astype(float)

train_df = df_model[df_model["year"].isin(TRAIN_YEARS)].copy()
eval_df  = df_model[df_model["year"] == EVAL_YEAR].copy()
test_df  = df_model[df_model["year"] == TEST_YEAR].copy()

X_train = train_df[num_cols].apply(pd.to_numeric, errors="coerce")
y_train = train_df[TARGET].astype(float)

X_eval = eval_df[num_cols].apply(pd.to_numeric, errors="coerce")
y_eval = eval_df[TARGET].astype(float)

X_test = test_df[num_cols].apply(pd.to_numeric, errors="coerce")
y_test = test_df[TARGET].astype(float)

print("Splits:",
      "train", X_train.shape,
      "eval", X_eval.shape,
      "test", X_test.shape,
      "| n_features", len(num_cols))

Splits: train (80, 10) eval (20, 10) test (20, 10) | n_features 10


In [29]:
# ===========================
# Cell 5 — Metrics helpers
# ===========================
def calc_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    sp = float(spearmanr(y_true, y_pred).correlation)
    return {"mae": mae, "rmse": rmse, "spearman": sp}

OPTIM_MODE = "rmse"  # minimize RMSE on eval
def score_for_optim(y_true, y_pred):
    m = calc_metrics(y_true, y_pred)
    return m["rmse"]

In [32]:
## ======================================
# Cell 6 — Optuna objective + run
# ======================================
SEED = 42
N_TRIALS = 40

dtrain = xgb.DMatrix(X_train, label=y_train)
deval  = xgb.DMatrix(X_eval,  label=y_eval)

def objective(trial: optuna.Trial):
    params = {
        # core booster params
        "eta": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.1, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 10.0),
        "lambda": trial.suggest_float("reg_lambda", 1e-4, 50.0, log=True),
        "alpha": trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True),

        # fixed
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "seed": SEED,
        "tree_method": "hist",
    }

    num_boost_round = trial.suggest_int("n_estimators", 200, 1400, step=100)

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=num_boost_round,
        evals=[(deval, "eval")],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    # predict using best iteration if available
    best_iter = getattr(booster, "best_iteration", None)
    if best_iter is not None:
        pred_eval = booster.predict(deval, iteration_range=(0, best_iter + 1))
    else:
        pred_eval = booster.predict(deval)

    score = score_for_optim(y_eval, pred_eval)

    m = calc_metrics(y_eval, pred_eval)
    trial.set_user_attr("eval_mae", m["mae"])
    trial.set_user_attr("eval_rmse", m["rmse"])
    trial.set_user_attr("eval_spearman", m["spearman"])
    trial.set_user_attr("best_iteration", best_iter)

    return score

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best score:", study.best_value)
print("Best params:", study.best_params)

[I 2026-02-25 13:09:43,688] A new study created in memory with name: no-name-0d6e182c-6f55-4565-9d67-e564ace2ed11
Best trial: 1. Best value: 4.50026:  10%|█         | 4/40 [00:00<00:01, 21.80it/s]

[I 2026-02-25 13:09:43,750] Trial 0 finished with value: 4.8395974028439745 and parameters: {'learning_rate': 0.030710573677773714, 'max_depth': 10, 'min_child_weight': 4.83437145318464, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'gamma': 1.5599452033620265, 'reg_lambda': 0.0002142973316927139, 'reg_alpha': 1.156732719914599, 'n_estimators': 900}. Best is trial 0 with value: 4.8395974028439745.
[I 2026-02-25 13:09:43,782] Trial 1 finished with value: 4.500260360232703 and parameters: {'learning_rate': 0.08341106432362087, 'max_depth': 3, 'min_child_weight': 17.052641538983092, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'gamma': 1.8182496720710062, 'reg_lambda': 0.0011097286548581117, 'reg_alpha': 0.000134801802908908, 'n_estimators': 800}. Best is trial 1 with value: 4.500260360232703.
[I 2026-02-25 13:09:43,824] Trial 2 finished with value: 4.820501269183085 and parameters: {'learning_rate': 0.03647316284911211, 'max_depth': 

Best trial: 1. Best value: 4.50026:  18%|█▊        | 7/40 [00:00<00:01, 22.41it/s]

[I 2026-02-25 13:09:43,955] Trial 5 finished with value: 4.724496864433846 and parameters: {'learning_rate': 0.07277150634170934, 'max_depth': 5, 'min_child_weight': 1.572867419497858, 'subsample': 0.7733551396716398, 'colsample_bytree': 0.5924272277627636, 'gamma': 9.695846277645586, 'reg_lambda': 2.6149218241872196, 'reg_alpha': 3.7713131110779936, 'n_estimators': 1300}. Best is trial 1 with value: 4.500260360232703.
[I 2026-02-25 13:09:43,997] Trial 6 finished with value: 4.851078539363148 and parameters: {'learning_rate': 0.059963338824126605, 'max_depth': 10, 'min_child_weight': 0.15981734133214875, 'subsample': 0.5979914312095727, 'colsample_bytree': 0.522613644455269, 'gamma': 3.2533033076326436, 'reg_lambda': 0.016408172581661183, 'reg_alpha': 7.933105363733024e-05, 'n_estimators': 1200}. Best is trial 1 with value: 4.500260360232703.
[I 2026-02-25 13:09:44,094] Trial 7 finished with value: 4.5567115025460865 and parameters: {'learning_rate': 0.02911701023242742, 'max_depth': 5

Best trial: 11. Best value: 4.38952:  30%|███       | 12/40 [00:00<00:01, 19.27it/s]

...

[I 2026-02-25 13:09:45,797] Trial 32 finished with value: 4.344831490308304 and parameters: {'learning_rate': 0.011698333011078508, 'max_depth': 9, 'min_child_weight': 11.316638662730945, 'subsample': 0.6225496560990353, 'colsample_bytree': 0.8326210038223775, 'gamma': 9.983055277344166, 'reg_lambda': 6.11800043307, 'reg_alpha': 0.00015674325206496706, 'n_estimators': 200}. Best is trial 18 with value: 4.109304682789168.
[I 2026-02-25 13:09:45,885] Trial 33 finished with value: 4.3366552617786205 and parameters: {'learning_rate': 0.016880341679993575, 'max_depth': 8, 'min_child_weight': 13.221201481509903, 'subsample': 0.5580449474245316, 'colsample_bytree': 0.8681604906557757, 'gamma': 7.7948292573896705, 'reg_lambda': 0.341030351739613, 'reg_alpha': 8.232926330341662e-06, 'n_estimators': 300}. Best is trial 18 with value: 4.109304682789168.


Best trial: 18. Best value: 4.1093:  92%|█████████▎| 37/40 [00:02<00:00, 12.10it/s]

[I 2026-02-25 13:09:45,987] Trial 34 finished with value: 4.1250406637697035 and parameters: {'learning_rate': 0.016465664250335434, 'max_depth': 7, 'min_child_weight': 6.3429975199440385, 'subsample': 0.5555522700282238, 'colsample_bytree': 0.9772967984617106, 'gamma': 7.87240435792225, 'reg_lambda': 0.28013214012403137, 'reg_alpha': 8.326847539329918e-06, 'n_estimators': 400}. Best is trial 18 with value: 4.109304682789168.
[I 2026-02-25 13:09:46,063] Trial 35 finished with value: 4.207317645381925 and parameters: {'learning_rate': 0.027697893337160752, 'max_depth': 7, 'min_child_weight': 5.773025591494742, 'subsample': 0.5430898923244905, 'colsample_bytree': 0.9610673316462865, 'gamma': 3.695914202425522, 'reg_lambda': 0.07521391525983098, 'reg_alpha': 1.3121975816503044e-06, 'n_estimators': 700}. Best is trial 18 with value: 4.109304682789168.
[I 2026-02-25 13:09:46,135] Trial 36 finished with value: 4.386090096929104 and parameters: {'learning_rate': 0.028923064461466686, 'max_dep

Best trial: 18. Best value: 4.1093: 100%|██████████| 40/40 [00:02<00:00, 14.88it/s]

[I 2026-02-25 13:09:46,208] Trial 37 finished with value: 4.244173810777112 and parameters: {'learning_rate': 0.03629876442551651, 'max_depth': 6, 'min_child_weight': 6.05587015878336, 'subsample': 0.5949594398609022, 'colsample_bytree': 0.9841108908151044, 'gamma': 4.193526423780509, 'reg_lambda': 0.006142341584677172, 'reg_alpha': 0.005066379277934959, 'n_estimators': 900}. Best is trial 18 with value: 4.109304682789168.
[I 2026-02-25 13:09:46,294] Trial 38 finished with value: 4.256950495070916 and parameters: {'learning_rate': 0.023282915880184153, 'max_depth': 7, 'min_child_weight': 5.500588300924541, 'subsample': 0.5724416339318958, 'colsample_bytree': 0.9002912394269258, 'gamma': 3.1162233966391324, 'reg_lambda': 0.07568676725993549, 'reg_alpha': 3.5084129982544526e-06, 'n_estimators': 700}. Best is trial 18 with value: 4.109304682789168.
[I 2026-02-25 13:09:46,373] Trial 39 finished with value: 4.479220248350037 and parameters: {'learning_rate': 0.025723585484087482, 'max_depth

In [33]:
# ======================================
# Cell 7 — Train best model + evaluate
# ======================================
best = study.best_params

# convert optuna params -> xgb params (note: lambda/alpha keys)
best_xgb_params = {
    "eta": best["learning_rate"],
    "max_depth": best["max_depth"],
    "min_child_weight": best["min_child_weight"],
    "subsample": best["subsample"],
    "colsample_bytree": best["colsample_bytree"],
    "gamma": best["gamma"],
    "lambda": best["reg_lambda"],
    "alpha": best["reg_alpha"],

    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "seed": 42,
    "tree_method": "hist",
}

num_boost_round = best["n_estimators"]

dtrain = xgb.DMatrix(X_train, label=y_train)
deval  = xgb.DMatrix(X_eval,  label=y_eval)

best_booster = xgb.train(
    params=best_xgb_params,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    evals=[(deval, "eval")],
    early_stopping_rounds=50,
    verbose_eval=False
)

best_iter = getattr(best_booster, "best_iteration", None)
if best_iter is not None:
    pred_eval = best_booster.predict(deval, iteration_range=(0, best_iter + 1))
else:
    pred_eval = best_booster.predict(deval)

print("Best iteration:", best_iter)
print("Eval metrics:", calc_metrics(y_eval, pred_eval))

Best iteration: 122
Eval metrics: {'mae': 3.5067500829696656, 'rmse': 4.109304682789168, 'spearman': 0.7020318014449615}


In [34]:
# ======================================
# Cell 8 — Fit on train+eval, predict test
# ======================================
X_train_full = pd.concat([X_train, X_eval], axis=0)
y_train_full = pd.concat([y_train, y_eval], axis=0)

dtrain_full = xgb.DMatrix(X_train_full, label=y_train_full)
dtest       = xgb.DMatrix(X_test)

final_booster = xgb.train(
    params=best_xgb_params,
    dtrain=dtrain_full,
    num_boost_round=(best_iter + 1) if best_iter is not None else num_boost_round,
    verbose_eval=False
)

pred_test = final_booster.predict(dtest)
print("Test metrics:", calc_metrics(y_test, pred_test))

Test metrics: {'mae': 3.661785674095154, 'rmse': 4.960741329840325, 'spearman': 0.7694622582512833}


In [35]:
from pathlib import Path
ART_DIR = Path("artifacts")
ART_DIR.mkdir(exist_ok=True, parents=True)

final_booster.save_model(str(ART_DIR / "xgb_best.json"))
print("Saved:", ART_DIR / "xgb_best.json")

Saved: artifacts\xgb_best.json
